In [2]:
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import joblib

# 1) Files saved from training
MODEL_DIR = Path('alex5050_model_NN')
MODEL_PATH = MODEL_DIR / 'best_alex5050_model.pth'
SCALER_PATH = MODEL_DIR / 'alex5050_scaler.joblib'

# 2) Feature order expected by alex_binary_5050 dataset (must match training)
FEATURE_COLUMNS = [
    'HighBP', 'HighChol', 'CholCheck', 'BMI', 'Smoker', 'Stroke',
    'HeartDiseaseorAttack', 'PhysActivity', 'Fruits', 'Veggies',
    'HvyAlcoholConsump', 'AnyHealthcare', 'NoDocbcCost', 'GenHlth',
    'MentHlth', 'PhysHlth', 'DiffWalk', 'Sex', 'Age', 'Education', 'Income'
]

# 3) Same network used during training
class DiabetesNet(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 32),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 1),
        )

    def forward(self, x):
        return self.net(x)

# 4) Load scaler and model
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
scaler = joblib.load(SCALER_PATH)

if hasattr(scaler, 'n_features_in_') and scaler.n_features_in_ != len(FEATURE_COLUMNS):
    raise ValueError(
        f'Feature count mismatch. Scaler expects {scaler.n_features_in_}, '
        f'but FEATURE_COLUMNS has {len(FEATURE_COLUMNS)}.'
    )

model = DiabetesNet(input_dim=len(FEATURE_COLUMNS)).to(device)
state_dict = torch.load(MODEL_PATH, map_location=device)
model.load_state_dict(state_dict)
model.eval()

# 5) Prediction helper
def predict_diabetes(instance_dict, threshold=0.37):
    row = pd.DataFrame([instance_dict])
    missing = [c for c in FEATURE_COLUMNS if c not in row.columns]
    extra = [c for c in row.columns if c not in FEATURE_COLUMNS]

    if missing:
        raise ValueError(f'Missing features: {missing}')
    if extra:
        print(f'Ignoring extra fields: {extra}')

    X = row[FEATURE_COLUMNS].astype(np.float32).values
    X_scaled = scaler.transform(X)

    with torch.no_grad():
        logits = model(torch.tensor(X_scaled, dtype=torch.float32).to(device))
        prob = torch.sigmoid(logits).cpu().numpy().ravel()[0]

    pred = int(prob >= threshold)
    label = 'Diabetes' if pred == 1 else 'No Diabetes'
    return {
        'probability_diabetes': float(prob),
        'prediction': pred,
        'label': label,
        'threshold_used': float(threshold),
    }


In [7]:
# 1) Likely Diabetes
diabetes_example = {
    'HighBP': 1, 'HighChol': 1, 'CholCheck': 1, 'BMI': 36.0, 'Smoker': 1,
    'Stroke': 0, 'HeartDiseaseorAttack': 1, 'PhysActivity': 0, 'Fruits': 0, 'Veggies': 0,
    'HvyAlcoholConsump': 0, 'AnyHealthcare': 1, 'NoDocbcCost': 1, 'GenHlth': 5,
    'MentHlth': 12, 'PhysHlth': 20, 'DiffWalk': 1, 'Sex': 1, 'Age': 11, 'Education': 3, 'Income': 2
}

# 2) Likely Non-Diabetes
non_diabetes_example = {
    'HighBP': 0, 'HighChol': 0, 'CholCheck': 1, 'BMI': 22.0, 'Smoker': 0,
    'Stroke': 0, 'HeartDiseaseorAttack': 0, 'PhysActivity': 1, 'Fruits': 1, 'Veggies': 1,
    'HvyAlcoholConsump': 0, 'AnyHealthcare': 1, 'NoDocbcCost': 0, 'GenHlth': 1,
    'MentHlth': 0, 'PhysHlth': 0, 'DiffWalk': 0, 'Sex': 0, 'Age': 4, 'Education': 6, 'Income': 7
}

# 3) Borderline (near decision threshold, ideally around 0.37)

borderline_example = {
    'HighBP': 1, 'HighChol': 0, 'CholCheck': 1, 'BMI': 28.0, 'Smoker': 0,
    'Stroke': 0, 'HeartDiseaseorAttack': 0, 'PhysActivity': 0, 'Fruits': 0, 'Veggies': 1,
    'HvyAlcoholConsump': 0, 'AnyHealthcare': 1, 'NoDocbcCost': 0, 'GenHlth': 3,
    'MentHlth': 2, 'PhysHlth': 3, 'DiffWalk': 0, 'Sex': 1, 'Age': 7, 'Education': 5, 'Income': 4
}


for name, sample in [
    ("Diabetes", diabetes_example),
    ("Non-Diabetes", non_diabetes_example),
    ("Borderline", borderline_example),
]:
    print(name, predict_diabetes(sample, threshold=0.37))

Diabetes {'probability_diabetes': 0.904675304889679, 'prediction': 1, 'label': 'Diabetes', 'threshold_used': 0.37}
Non-Diabetes {'probability_diabetes': 0.011291437782347202, 'prediction': 0, 'label': 'No Diabetes', 'threshold_used': 0.37}
Borderline {'probability_diabetes': 0.49424588680267334, 'prediction': 1, 'label': 'Diabetes', 'threshold_used': 0.37}
